## Baselines

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def sample_bimodal_beta(n=5000, p=0.5, a=2, b=90):
    """
    Generate samples from a bimodal distribution on [0,1] using a mixture of two Beta distributions.

    With probability p:
        sample from Beta(a, b) → concentrated near 0
    Otherwise:
        sample from Beta(b, a) → concentrated near 1

    This creates a bimodal shape with peaks near 0 and 1.
    """
    xs = []
    for _ in range(n):
        if random.random() < p:
            xs.append(random.betavariate(a, b))
        else:
            xs.append(random.betavariate(b, a))
    return xs

In [ ]:
def generate_vectors(num_classes, no_of_examples):
    """
    Generate unique normalized proportion vectors.

    - Each vector sums to 1
    - Avoids duplicates
    - Avoids already existing vectors from dataset
    """

    seen = set()

    vectors = []

    while len(vectors) < no_of_examples:
        
        # Sample raw values from bimodal beta
        v = sample_bimodal_beta(n=num_classes, p=0.5, a=1, b=14)

        # Normalize to sum to 1
        s = sum(v)
        v = tuple(round(x / s, 3) for x in v)

        # Fix rounding drift (important)
        diff = 1.0 - sum(v)
        v = list(v)
        v[0] = round(v[0] + diff, 3)
        v = tuple(v)

        # Validate
        if v in seen or min(v) < 0.01 or sum(v)!=1:
            continue

        seen.add(v)
        vectors.append(v)

    return vectors

### Baseline 1  (centroid guess)

##### have 1/no_of_classes as the prediction and sample from the distribution that we usually use and compute the MAE

In [ ]:
num_classes = 3
num_examples = 800

In [ ]:
def distance_of_proportions(X, no_of_classes, ddof=0):
    """
    X: array shape (N,no_of_classes) with values in [0,1]
    
    returns:
    P: normalized proportions, shape (N,no_of_classes)
    """
    X = np.asarray(X, dtype=float)
    l1 = np.mean(np.abs(X - float(1/no_of_classes)), axis=1)
    return l1

In [ ]:
def plot_histogram(values, bins=200, title="Histogram"):
    """
    Plot a histogram from a list of values.

    Args:
        values (list or array): Input data.
        bins (int): Number of bins.
        title (str): Plot title.
    """
    plt.figure()
    plt.hist(values, bins=bins)
    plt.title(title)
    plt.xlabel("Value")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
vectors = generate_vectors(num_classes, num_examples)
l1 = distance_of_proportions(vectors, num_classes)

In [ ]:
mean_MAE = sum(l1)/len(l1)
print(mean_MAE)

In [ ]:
plot_histogram(l1)

### Baseline 2 

#####  draw from the distribution once and draw again and compute the MAE

In [ ]:
num_examples = 800

In [ ]:
# each vector geenration process even when generating multiple vectors at once is indepdnet of each other , so we dont need nultiple 
# generate vector calls actually. 

l1 = []
for i in range(num_examples):
    dev_arr = []
    proportion_arr_1 = generate_vectors(num_classes, 1)[0]
    proportion_arr_2 = generate_vectors(num_classes, 1)[0]
    for i in range(len(proportion_arr_1)):
        dev_arr.append(abs(proportion_arr_2[i]-proportion_arr_1[i]))
    l1.append(sum(dev_arr)/len(dev_arr))

In [ ]:
mean_MAE = sum(l1)/len(l1)
print(mean_MAE)

In [ ]:
plot_histogram(l1)

### Baseline 3 

#####  the finetuned indices that we have is constant for every experiment...we then select the seed_indices , compute the inteprolations again, and then compute the alignment scores again, with different increasing overlap. 

#####  we put the entire alignment score row of the seed select datasets into the fcn, with the label whether this was in the fientuning set or not 

##### then do inference on every training point to check if its fientuned or not but then we would have to compute alignment score for every pseudoexpert for a specific itnerplation for each point

#####   then compute mae on the proportion that we get the from the set of points marked as fientuned. 